# Object Detection Evaluation Metrics

Companion to [Overview of Object Detection Evaluation Metrics](https://todatabeyond.com/blog/overview-of-object-detection-evaluation-metrics).

Run all cells in order on CPU. No API keys, datasets, model downloads or GPU are required. Predictions below are explicit teaching fixtures, not a trained detector benchmark. The metric implementation uses TorchMetrics with the official COCO evaluation backend. FPS depends on the full inference pipeline and hardware; this notebook does not manufacture a detector FPS benchmark.


## Setup

These pinned versions were executed with Python 3.13 on CPU. Use Python 3.10–3.13 locally, or a compatible Colab runtime. Installation may require a kernel restart if a notebook session has already imported different package versions.


In [ ]:
%pip install --quiet torch==2.10.0 torchvision==0.25.0 torchmetrics==1.9.0 pycocotools==2.0.11


## 1. Intersection over union

Use continuous `xyxy` pixel coordinates. The second prediction covers 24 × 24 pixels inside a 30 × 30 target, so IoU is 576/900 = 0.64. Zero-union boxes return zero in this teaching helper.


In [ ]:
def intersection_over_union(a, b):
    """IoU for valid, axis-aligned xyxy boxes (continuous coordinates)."""
    if any(box[2] < box[0] or box[3] < box[1] for box in (a, b)):
        raise ValueError("Box maxima must not be smaller than minima")
    width = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    height = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    intersection = width * height
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0

print(f"Class 0 IoU: {intersection_over_union([0, 0, 10, 10], [0, 0, 10, 10]):.4f}")
print(f"Class 1 IoU: {intersection_over_union([40, 40, 64, 64], [40, 40, 70, 70]):.4f}")


## 2. COCO-style mean average precision

Each list entry represents one image. Predicted boxes, scores and integer class labels must have matching lengths; targets require boxes and class labels. The default evaluator averages over IoU thresholds 0.50 through 0.95 in increments of 0.05 and uses COCO interpolation.

Class 0 is perfect at all ten thresholds. Class 1 matches at 0.50, 0.55 and 0.60 only, so its AP averaged over those thresholds is 0.30. The two-class mAP is (1.00 + 0.30)/2 = 0.65. mAP@0.50 is 1.00 and mAP@0.75 is 0.50. This is not interchangeable with a single-IoU or VOC AP convention.


In [ ]:
import torch
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# One image, two classes. Boxes use [xmin, ymin, xmax, ymax] pixels.
predictions = [{
    "boxes": torch.tensor([[0., 0., 10., 10.], [40., 40., 64., 64.]]),
    "scores": torch.tensor([0.95, 0.90]),
    "labels": torch.tensor([0, 1], dtype=torch.int64),
}]
ground_truth = [{
    "boxes": torch.tensor([[0., 0., 10., 10.], [40., 40., 70., 70.]]),
    "labels": torch.tensor([0, 1], dtype=torch.int64),
}]

metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
metric.update(predictions, ground_truth)
results = metric.compute()
print(f"mAP@[0.50:0.95]: {results['map'].item():.4f}")
print(f"mAP@0.50: {results['map_50'].item():.4f}")
print(f"mAP@0.75: {results['map_75'].item():.4f}")


## 3. Regression checks

Check overlap boundaries, invalid box order, perfect detections, high-confidence false positives, duplicate detections, empty predictions and wrong classes. A model adapter should produce the same per-image schema; the evaluator must see every image in the evaluation split.


In [ ]:
from copy import deepcopy
from math import isclose

assert isclose(intersection_over_union([0, 0, 10, 10], [0, 0, 10, 10]), 1.0)
assert intersection_over_union([0, 0, 10, 10], [20, 20, 30, 30]) == 0.0
assert intersection_over_union([0, 0, 0, 0], [0, 0, 0, 0]) == 0.0
assert isclose(intersection_over_union([40, 40, 64, 64], [40, 40, 70, 70]), 0.64)
try:
    intersection_over_union([10, 0, 0, 10], [0, 0, 10, 10])
except ValueError:
    pass
else:
    raise AssertionError("Inverted boxes must be rejected")

assert isclose(results["map"].item(), 0.65, abs_tol=1e-6)
assert isclose(results["map_50"].item(), 1.0, abs_tol=1e-6)
assert isclose(results["map_75"].item(), 0.5, abs_tol=1e-6)

def evaluate(preds, targets):
    evaluator = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    evaluator.update(preds, targets)
    return evaluator.compute()

# Perfect detections score one at every IoU threshold.
perfect = deepcopy(predictions)
perfect[0]["boxes"] = ground_truth[0]["boxes"].clone()
assert isclose(evaluate(perfect, ground_truth)["map"].item(), 1.0, abs_tol=1e-6)

# A high-confidence background false positive lowers class-0 AP.
with_fp = deepcopy(predictions)
with_fp[0]["boxes"] = torch.cat([torch.tensor([[100., 100., 110., 110.]]), with_fp[0]["boxes"]])
with_fp[0]["scores"] = torch.tensor([0.99, 0.95, 0.90])
with_fp[0]["labels"] = torch.tensor([0, 0, 1], dtype=torch.int64)
assert isclose(evaluate(with_fp, ground_truth)["map"].item(), 0.4, abs_tol=1e-6)

# Duplicate boxes do not create another true positive for the same object.
duplicate = deepcopy(perfect)
duplicate[0]["boxes"] = torch.cat([duplicate[0]["boxes"], duplicate[0]["boxes"][:1]])
duplicate[0]["scores"] = torch.tensor([0.95, 0.90, 0.10])
duplicate[0]["labels"] = torch.tensor([0, 1, 0], dtype=torch.int64)
assert isclose(evaluate(duplicate, ground_truth)["map"].item(), 1.0, abs_tol=1e-6)

empty = [{"boxes": torch.empty((0, 4)), "scores": torch.empty((0,)),
          "labels": torch.empty((0,), dtype=torch.int64)}]
assert evaluate(empty, ground_truth)["map"].item() == 0.0

wrong_classes = deepcopy(perfect)
wrong_classes[0]["labels"] = torch.tensor([1, 0], dtype=torch.int64)
assert evaluate(wrong_classes, ground_truth)["map"].item() == 0.0
print("All IoU and COCO mAP regression checks passed.")


## References

- [TorchMetrics MeanAveragePrecision](https://lightning.ai/docs/torchmetrics/stable/detection/mean_average_precision)
- [COCO evaluation implementation](https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py)

No outputs or execution counts are committed.
